# Plotting Quil pulse schedules

This notebook plots a Quil program's pulse schedule with both of `quil_plotting`'s backends:

- **plotly** (`plot_schedule`) — the original backend.
- **Altair / Vega-Lite** (`plot_schedule_altair`) — renders static images in process, so it does
  not need an installed copy of Chrome to write a png or svg.

Both read the same schedule dataframe and take the same options, so you can compare them directly.

## Setup

Locate the package and the bundled test programs. This works without installing `quil_plotting`,
as long as the notebook lives somewhere inside the package.

In [ ]:
import sys
from pathlib import Path

PACKAGE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "quil_plotting").is_dir())
sys.path.insert(0, str(PACKAGE_ROOT))

PROGRAM_DIR = PACKAGE_ROOT / "tests" / "programs"

# Imported after the path is set up, hence the `noqa`.
from quil.program import Program  # noqa: E402

from quil_plotting import (  # noqa: E402
    add_plot_metadata,
    plot_schedule,
    plot_schedule_altair,
    program_to_dataframe,
)

sorted(path.stem for path in PROGRAM_DIR.glob("*.quil"))

## Choose a program

Set `PROGRAM_NAME` to any of the names listed above and re-run the notebook. Some suggestions:

| name | what it shows |
| --- | --- |
| `multiple_gates_with_measures` | small and quick — a good place to start |
| `single_gate_iswap` | one two-qubit gate, with the coupler frame alongside the qubits |
| `sequenced_hadamard_barrier` | how barriers serialize the schedule |
| `randomized_circuit_0` | a busier circuit, ~26 us long |
| `rotated-surface-code` | 41 frames over 8 rounds of syndrome extraction — the stress case |

In [ ]:
PROGRAM_NAME = "multiple_gates_with_measures"

program = Program.parse((PROGRAM_DIR / f"{PROGRAM_NAME}.quil").read_text())
print(f"{PROGRAM_NAME}: {len(program.body_instructions)} instructions, "
      f"{len(program.calibrations.calibrations)} calibrations, {len(program.frames)} frames")

## The schedule dataframe

Both backends are built on this. Each row is one IQ sample of one pulse, carrying the metadata
needed to place, colour, and label it. `program_to_dataframe` expands the program's calibrations
and schedules the result; `add_plot_metadata` adds the plotting columns (`Offset`, `Color`,
`Normalized IQ`, `Label`, ...).

In [ ]:
df = add_plot_metadata(program_to_dataframe(program))
print(f"{len(df)} rows, spanning {(df['Time (s)'].max() - df['Time (s)'].min()) * 1e6:.3f} us")
df.head()

## The plotly backend

**Interacting:** drag to zoom into a time range, double-click to reset, and click a legend entry to
hide that operation (double-click one to isolate it). Hovering a pulse shows its I/Q value, frame, and
channel type.

In [ ]:
fig = plot_schedule(program)
fig.show()

## The Altair backend

The same schedule, rendered through Vega-Lite.

**Interacting:** drag to pan and scroll to zoom (both axes), and click a legend entry to isolate
that operation — the others dim rather than disappear, so the shape of the schedule is preserved.
Shift-click to select several. Hovering a pulse shows its I/Q value, frame, and channel type.

Altair's default renderer loads Vega from a CDN. On a machine without internet access, run
`alt.renderers.enable("mimetype")` first and JupyterLab will render the chart itself.

In [ ]:
chart = plot_schedule_altair(program)
chart

## Static export

Altair renders png and svg in process. Plotly's `write_image` goes through Kaleido, which needs a
copy of Chrome on the machine — if it is missing, install one with `plotly_get_chrome`.

In [ ]:
output_dir = Path("plots")
output_dir.mkdir(exist_ok=True)

chart.save(output_dir / f"{PROGRAM_NAME}-altair.svg")
chart.save(output_dir / f"{PROGRAM_NAME}-altair.png", ppi=100)

# Needs Chrome; comment out if it is not installed.
fig.write_image(output_dir / f"{PROGRAM_NAME}-plotly.png")

sorted(path.name for path in output_dir.iterdir())

## Options

Both backends accept the same arguments, so the same call works against either. A few worth trying:

- `runners` — what to stack up the y-axis (default `"Qubit"`).
- `color_by` / `label_by` — what drives the colours and the legend. Defaults to `"Operation"`,
  the gate (or, in future, reset) each pulse came from. `"Channel Type"` groups by
  hardware channel instead of by operation.
- `exclude_readout` — set `False` to include the readout pulses, which are long and flat and
  otherwise dominate the normalization.
- `normalize_by` — the grouping whose peak amplitude scales each pulse to fit its row.

In [ ]:
plot_schedule_altair(
    program,
    color_by="Channel Type",
    label_by="Channel Type",
    exclude_readout=False,
)